In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# CELL 1 — Mount Google Drive & Setup QGenesis
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/QGenesis', exist_ok=True)
print('Drive ready - cyril_memory.txt saves to Drive/QGenesis/')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive ready - cyril_memory.txt saves to Drive/QGenesis/


### CELL 3 — Exposing Cyril as a Web API
To connect Cyril to your website, we need to create a secure endpoint that your site can 'talk' to. We will use Flask for the API and Ngrok to create a tunnel to the web.

In [ ]:
import os, json, datetime
from transformers import pipeline

MEMORY_PATH = '/content/drive/MyDrive/QGenesis/cyril_memory.txt'
os.makedirs(os.path.dirname(MEMORY_PATH), exist_ok=True) # Ensure directory exists at cell execution

def load_memory():
    if os.path.exists(MEMORY_PATH):
        with open(MEMORY_PATH, 'r') as f:
            return f.read()
    return ""

def save_memory(entry):
    # os.makedirs(os.path.dirname(MEMORY_PATH), exist_ok=True) # Redundant here now, but harmless
    with open(MEMORY_PATH, 'a') as f:
        timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        f.write(f"\n[{timestamp}]{entry}")

# --- Cyril Boot Sequence ---
memory = load_memory()
print("=" * 50)
print("Hello Joseph. Cyril is online.")
if memory:
    print("Memory restored. QGenesis systems nominal.")
else:
    print("No prior memory found. Initializing fresh session.")
print("What shall we build today?")
print("=" * 50)

# --- Install dependencies ---
import subprocess
subprocess.run(["pip", "install", "-q", "requests", "transformers"], check=True)
import requests

# --- Initialize Hugging Face Model ---
HF_MODEL_NAME = "Mastercyril1/Cyril-QAI"
try:
    # Using 'text-generation' pipeline. Adjust parameters as needed for specific model behavior.
    cyril_generator = pipeline("text-generation", model=HF_MODEL_NAME)
    print(f"Hugging Face model '{HF_MODEL_NAME}' loaded successfully.")
except Exception as e:
    print(f"Error loading Hugging Face model: {e}")
    print("Note: This model is gated. If you have access, run `!huggingface-cli login` in a separate cell and provide your token.")
    cyril_generator = None # Fallback if model loading fails

# --- Web Search Tool ---
def cyril_search(query):
    try:
        url = "https://api.duckduckgo.com/"
        params = {"q": query, "format": "json", "no_redirect": 1, "no_html": 1}
        r = requests.get(url, params=params, timeout=10)
        data = r.json()
        results = []
        if data.get("AbstractText"):
            results.append(data["AbstractText"])
        for topic in data.get("RelatedTopics", [])[:3]:
            if isinstance(topic, dict) and topic.get("Text"):
                results.append(topic["Text"])
        return "\n".join(results) if results else "No results found."
    except Exception as e:
        return f"Search error: {e}"

# --- Cyril Chat Loop ---
def cyril_respond(user_input):
    user_lower = user_input.lower()

    if any(kw in user_lower for kw in ["search", "look up", "find info", "what is", "who is"]):
        query = user_input.replace("search", "").replace("look up", "").strip()
        result = cyril_search(query)
        response = f"[Cyril Search]: {result}"
    elif "memory" in user_lower:
        mem = load_memory()
        response = f"[Memory Log]:\n{mem}" if mem else "[Memory is currently empty.]"
    elif "hello" in user_lower or "hi" in user_lower:
        response = "Hello Joseph. Cyril systems online. QGenesis is active."
    elif "status" in user_lower:
        response = "All QGenesis systems nominal. Drive memory active. Ready to build."
    else:
        # Use Hugging Face model for general responses if loaded
        if cyril_generator:
            try:
                # Generate response using the HF model
                # max_new_tokens can be adjusted to control response length
                # The model might return the prompt as part of the generated text, so we try to extract the new part.
                generated_text = cyril_generator(user_input, max_new_tokens=50, num_return_sequences=1)[0]['generated_text']
                if generated_text.startswith(user_input):
                    response = f"[Cyril AI]: {generated_text[len(user_input):].strip()}"
                else:
                    response = f"[Cyril AI]: {generated_text.strip()}"
            except Exception as e:
                print(f"Error generating response with Hugging Face model: {e}")
                response = f"[Cyril]: Processing '{user_input}' — QGenesis AI ready for your command, Joseph. (HF model failed)"
        else:
            response = f"[Cyril]: Processing '{user_input}' — QGenesis AI ready for your command, Joseph. (HF model not loaded)"

    save_memory(f"USER: {user_input} | CYRIL: {response[:100]}")
    return response

# --- Interactive Input Loop ---
print("\nType your message below (or 'quit' to exit):\n")
while True:
    try:
        user_input = input("Joseph > ")
        if user_input.lower() in ["quit", "exit", "bye"]:
            print("Cyril signing off. Memory saved. Until next time, Joseph.")
            break
        if user_input.strip() == "":
            continue
        print(cyril_respond(user_input))
    except (EOFError, KeyboardInterrupt):
        print("\nCyril session ended.")
        break

Hello Joseph. Cyril is online.
No prior memory found. Initializing fresh session.
What shall we build today?
Error loading Hugging Face model: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/Mastercyril1/Cyril-QAI.
401 Client Error. (Request ID: Root=1-6a5c005f-5cca2379663cf36a592901b0;b5e554fa-8d1d-4c7b-89d5-f7f4ba7cafb3)

Cannot access gated repo for url https://huggingface.co/Mastercyril1/Cyril-QAI/resolve/main/config.json.
Access to model Mastercyril1/Cyril-QAI is restricted. You must have access to it and be authenticated to access it. Please log in.

Type your message below (or 'quit' to exit):



In [ ]:
# Run this cell to log in to Hugging Face
!huggingface-cli login

In [ ]:
from google.colab import userdata
userdata.get('secretName')

In [ ]:
!pip install -q flask flask-cors pyngrok

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

app = Flask(__name__)
CORS(app) # Enable Cross-Origin Resource Sharing for your website

@app.route('/ask', methods=['POST'])
def ask_cyril():
    data = request.json
    user_query = data.get('message', '')
    if not user_query:
        return jsonify({'error': 'No message provided'}), 400

    # Use the cyril_respond logic defined in Cell 2
    response = cyril_respond(user_query)
    return jsonify({'response': response})

print("Flask API defined. Ready to launch tunnel.")

#### Instructions to Connect:
1. Get a free Auth Token from [dashboard.ngrok.com](https://dashboard.ngrok.com/).
2. Run the cell below with your token to get a public URL.
3. In your website code, send POST requests to `[YOUR_NGROK_URL]/ask`.

In [ ]:
import os
from pyngrok import ngrok

# Your provided Ngrok Auth Token
NGROK_TOKEN = "3AgolfknGwCjbrhFsH7czbE6fjs_4PMA8BYdZkA12MbbKkWq8"
ngrok.set_auth_token(NGROK_TOKEN)

# Open a tunnel on port 5000
try:
    # Close existing tunnels to avoid session conflicts
    tunnels = ngrok.get_tunnels()
    for tunnel in tunnels:
        ngrok.disconnect(tunnel.public_url)

    public_url = ngrok.connect(5000).public_url
    print("=" * 50)
    print("SUCCESS: CYRIL TUNNEL ESTABLISHED")
    print(f"PUBLIC URL: {public_url}")
    print(f"API ENDPOINT: {public_url}/ask")
    print("=" * 50)
    print("Now copy the PUBLIC URL above and update the script in Cell 91b69468.")
except Exception as e:
    print(f"Error connecting to ngrok: {e}")

# Start the Flask server to listen for website requests
app.run(port=5000)

### CELL 4 — Website Integration Snippet
Copy and paste this code into the HTML of your `quantum-artificial-intelligence` page.

**Note:** You must replace `YOUR_NGROK_URL` in the script below with the live URL printed in the cell above.

In [ ]:
# This is a template for your website's frontend.
# You can use this to test the connection locally or embed it in your CMS.

html_code = """
<div id='cyril-chat-container'>
    <div id='chat-box' style='height: 300px; overflow-y: scroll; border: 1px solid #00f2ff; background: #000; color: #00f2ff; padding: 10px; font-family: monospace;'>
        [Cyril]: Quantum Systems Online. How can I assist with QGenesis today?
    </div>
    <input type='text' id='user-input' style='width: 80%; background: #111; color: #fff; border: 1px solid #00f2ff;' placeholder='Type message...'>
    <button onclick='sendMessage()' style='background: #00f2ff; color: #000;'>Send</button>
</div>

<script>
async function sendMessage() {
    const input = document.getElementById('user-input');
    const chatBox = document.getElementById('chat-box');
    const message = input.value;

    chatBox.innerHTML += `<br><b>Joseph:</b> ${message}`;
    input.value = '';

    // REPLACE THE URL BELOW WITH YOUR ACTUAL NGROK URL
    const response = await fetch('YOUR_NGROK_URL/ask', {
        method: 'POST',
        headers: {'Content-Type': 'application/json'},
        body: JSON.stringify({ message: message })
    });

    const data = await response.json();
    chatBox.innerHTML += `<br><b>Cyril:</b> ${data.response}`;
    chatBox.scrollTop = chatBox.scrollHeight;
}
</script>
"""

print("Integration code ready. Ensure you replace 'YOUR_NGROK_URL' with the public URL from the previous step.")

Once mounted, you can access your drive files under `/content/drive/MyDrive/`. Let's list the top-level files and folders to confirm.

In [ ]:
import os

drive_path = '/content/drive/MyDrive/'
try:
    files = os.listdir(drive_path)
    print("Files in Google Drive:")
    for f in files[:10]: # List first 10 items
        print(f)
except FileNotFoundError:
    print("Drive not found. Please ensure the mount step was successful.")